### **1. Setup (change DATA and OUT first)**

- Connect DuckDB and create "views" pointing at the five CSV files.
- A view is just a saved query, so nothing is loaded into memory yet.

In [1]:
import duckdb, os

DATA = os.path.expanduser("~/Desktop/Capstone_Project")        # folder with the five CSV files
OUT  = os.path.expanduser("~/Desktop/Capstone_Project/output")    # where the results get saved
os.makedirs(OUT, exist_ok=True)

con = duckdb.connect()                      # in-memory DuckDB database
con.sql("SET memory_limit='32GB'")          # cap memory use, since the server is shared
con.sql("SET threads=8")                    # use 8 CPU threads
con.sql(f"SET temp_directory='{OUT}/tmp'")  # where DuckDB spills data if memory runs out

def csv(name):
    # builds the DuckDB expression that reads one CSV file (first row = column names)
    return f"read_csv('{DATA}/{name}', header=true)"

# One view per file; the later SQL uses these names
con.sql(f"CREATE VIEW components AS SELECT * FROM {csv('connected_components.csv')}")
con.sql(f"CREATE VIEW nodes      AS SELECT * FROM {csv('nodes.csv')}")
con.sql(f"CREATE VIEW edges      AS SELECT * FROM {csv('edges.csv')}")
con.sql(f"CREATE VIEW bg_nodes   AS SELECT * FROM {csv('background_nodes.csv')}")
con.sql(f"CREATE VIEW bg_edges   AS SELECT * FROM {csv('background_edges.csv')}")

**Quick test** (LIMIT 3): 
- **Goal**: Read only 3 rows of the 83 GB file to confirm DuckDB can read it.

In [ ]:
con.sql("SELECT * FROM bg_edges LIMIT 3").show()

**Result**: It returned in under a second. clId1, clId2, txId, and feat#1 to feat#95.

### **2. Double-check the column names.**

- **Goal**: Print the real column names of each file to confirm they match the names used in the SQL below.

In [2]:
# The code below assumes: ccId, ccLabel, clId, clId1, clId2, txId. Fix the names if these differ.
for t in ["components", "nodes", "edges", "bg_nodes", "bg_edges"]:
    print(t)
    con.sql(f"SELECT column_name, column_type FROM (DESCRIBE {t}) LIMIT 6").show()

components
┌─────────────┬─────────────┐
│ column_name │ column_type │
│   varchar   │   varchar   │
├─────────────┼─────────────┤
│ ccId        │ BIGINT      │
│ ccLabel     │ VARCHAR     │
└─────────────┴─────────────┘

nodes
┌─────────────┬─────────────┐
│ column_name │ column_type │
│   varchar   │   varchar   │
├─────────────┼─────────────┤
│ clId        │ BIGINT      │
│ ccId        │ BIGINT      │
└─────────────┴─────────────┘

edges
┌─────────────┬─────────────┐
│ column_name │ column_type │
│   varchar   │   varchar   │
├─────────────┼─────────────┤
│ clId1       │ BIGINT      │
│ clId2       │ BIGINT      │
│ txId        │ BIGINT      │
└─────────────┴─────────────┘

bg_nodes
┌─────────────┬─────────────┐
│ column_name │ column_type │
│   varchar   │   varchar   │
├─────────────┼─────────────┤
│ clId        │ BIGINT      │
│ feat#1      │ BIGINT      │
│ feat#2      │ BIGINT      │
│ feat#3      │ BIGINT      │
│ feat#4      │ BIGINT      │
│ feat#5      │ BIGINT      │
└────

**Result**: The names match 
- `ccId` and `ccLabel` in components
- `clId` and `ccId` in nodes
- `clId1`, `clId2`, and `txId` in edges
- `clId`, `clId1`, `clId2`, `txId`, and the `feat#` columns in the background files.

### **3. Labeled nodes: ccId -> clId**

**Goal**: Link each labeled node (clId) to its component (ccId) and add the component’s label (licit or suspicious).

In [3]:
con.sql("""
CREATE TABLE labeled_nodes AS
SELECT n.ccId, c.ccLabel, n.clId
FROM nodes n
JOIN components c ON n.ccId = c.ccId
""")

con.sql("SELECT * FROM labeled_nodes LIMIT 5").show()
con.sql("SELECT COUNT(*) AS rows FROM labeled_nodes").show()

┌────────┬────────────┬───────────┐
│  ccId  │  ccLabel   │   clId    │
│ int64  │  varchar   │   int64   │
├────────┼────────────┼───────────┤
│  41121 │ licit      │ 515498410 │
│  27974 │ licit      │ 630366534 │
│ 108020 │ licit      │ 903790945 │
│   6544 │ suspicious │ 449108887 │
│  27234 │ licit      │ 877994419 │
└────────┴────────────┴───────────┘

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 444521 │
└────────┘



**Result**: `labeled_nodes`, with 444,521 rows (one per labeled node).

### **4. Labeled edges, with the component of each endpoint**

**Goal**: Build a table with one row per labeled edge, showing the component of each end of the edge. A left join keeps every edge, so problems show up in the checks later.

In [4]:
con.sql("""
CREATE TABLE labeled_edges AS
SELECT n1.ccId, c.ccLabel, e.clId1, e.clId2, e.txId, n2.ccId AS ccId_end
FROM edges e
LEFT JOIN nodes n1 ON e.clId1 = n1.clId
LEFT JOIN nodes n2 ON e.clId2 = n2.clId
LEFT JOIN components c ON n1.ccId = c.ccId
""")

con.sql("SELECT * FROM labeled_edges LIMIT 5").show()
con.sql("SELECT COUNT(*) AS rows FROM labeled_edges").show()

┌───────┬────────────┬───────────┬───────────┬───────────┬──────────┐
│ ccId  │  ccLabel   │   clId1   │   clId2   │   txId    │ ccId_end │
│ int64 │  varchar   │   int64   │   int64   │   int64   │  int64   │
├───────┼────────────┼───────────┼───────────┼───────────┼──────────┤
│  1737 │ licit      │ 653013230 │ 690139842 │ 562099376 │     1737 │
│  2490 │ licit      │ 725167799 │ 725359763 │ 587069143 │     2490 │
│  2490 │ licit      │ 723749687 │ 725167799 │ 587464070 │     2490 │
│ 84893 │ suspicious │ 750479564 │ 968621870 │ 587069686 │    84893 │
│ 63977 │ licit      │ 637736927 │ 654734522 │ 535922209 │    63977 │
└───────┴────────────┴───────────┴───────────┴───────────┴──────────┘

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 367137 │
└────────┘



**Result**: `labeled_edges`, with 367,137 rows (one per labeled edge).

### **5. Node features (scans the 5 GB file, takes a minute or more)**

**Goal**: Match each labeled node to its 43 background features in background_nodes.csv, keeping only the labeled nodes instead of the whole 49M-node graph.

In [5]:
con.sql("""
CREATE TABLE node_features AS
SELECT ln.*, b.* EXCLUDE (clId)
FROM labeled_nodes ln
JOIN bg_nodes b ON ln.clId = b.clId
""")

con.sql('SELECT ccId, ccLabel, clId, "feat#1", "feat#2", "feat#3" FROM node_features LIMIT 5').show()
con.sql("SELECT COUNT(*) AS rows FROM node_features").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────┬───────────┬────────┬────────┬────────┐
│ ccId  │ ccLabel │   clId    │ feat#1 │ feat#2 │ feat#3 │
│ int64 │ varchar │   int64   │ int64  │ int64  │ int64  │
├───────┼─────────┼───────────┼────────┼────────┼────────┤
│ 39995 │ licit   │ 878497135 │     29 │      8 │      0 │
│ 60707 │ licit   │ 727876364 │     41 │     26 │      0 │
│ 16765 │ licit   │ 741548098 │     34 │     17 │      0 │
│ 12664 │ licit   │ 705793401 │     32 │     13 │      0 │
│ 35613 │ licit   │ 153092543 │     43 │     27 │     33 │
└───────┴─────────┴───────────┴────────┴────────┴────────┘

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 444521 │
└────────┘



**Result**: `node_features`, with 444,521 rows and 46 columns (`ccId`, `ccLabel`, `clId`, and 43 features). Every labeled node matched once.

### **6. Edge features (scans the 83 GB file, so run this one last)**

**Goal**: Match each labeled edge to its 95 background features in background_edges.csv, using clId1, clId2, and txId. DuckDB reads the 83 GB file once and keeps only the matches.

In [6]:
con.sql("""
CREATE TABLE edge_features AS
SELECT le.*, b.* EXCLUDE (clId1, clId2, txId)
FROM labeled_edges le
JOIN bg_edges b ON le.clId1 = b.clId1 AND le.clId2 = b.clId2 AND le.txId = b.txId
""")

con.sql('SELECT ccId, ccLabel, clId1, clId2, txId, "feat#1", "feat#2", "feat#3" FROM edge_features LIMIT 5').show()
con.sql("SELECT COUNT(*) AS rows FROM edge_features").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────┬─────────┬───────────┬───────────┬───────────┬────────┬────────┬────────┐
│  ccId  │ ccLabel │   clId1   │   clId2   │   txId    │ feat#1 │ feat#2 │ feat#3 │
│ int64  │ varchar │   int64   │   int64   │   int64   │ int64  │ int64  │ int64  │
├────────┼─────────┼───────────┼───────────┼───────────┼────────┼────────┼────────┤
│  44825 │ licit   │  28527113 │ 742704048 │   3107601 │     25 │     68 │     41 │
│  90899 │ licit   │ 908136059 │ 599598370 │ 408941847 │     34 │     67 │     42 │
│ 106993 │ licit   │ 756404140 │   9857117 │   2727113 │     54 │     68 │     44 │
│   4368 │ licit   │ 742314487 │ 251344663 │ 598157325 │     35 │     68 │     43 │
│  39995 │ licit   │ 878497135 │ 727865454 │ 598905701 │     24 │     65 │     46 │
└────────┴─────────┴───────────┴───────────┴───────────┴────────┴────────┴────────┘

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 367137 │
└────────┘



**Result**: `edge_features`, with 367,137 rows and 101 columns (`ccId`, `ccLabel`, `clId1`, `clId2`, `txId`, `ccId_end`, and 95 features). Every labeled edge matched once.

### **7. Verify the joins**

**Goal**: Check that the row counts match the proposal and that the joins didn’t lose, duplicate, or mismatch any records.

In [7]:
checks = {
  "components (expect 121,810)": "SELECT COUNT(*) FROM components",
  "rows in nodes.csv (expect 444,521)": "SELECT COUNT(*) FROM nodes",
  "nodes after feature join (should match)": "SELECT COUNT(*) FROM node_features",
  "rows in edges.csv (expect 367,137)": "SELECT COUNT(*) FROM edges",
  "edges after feature join (should match)": "SELECT COUNT(*) FROM edge_features",
  "duplicate clId in node_features (expect 0)": "SELECT COUNT(*) - COUNT(DISTINCT clId) FROM node_features",
  "clId in more than one component (expect 0)": "SELECT COUNT(*) FROM (SELECT clId FROM nodes GROUP BY clId HAVING COUNT(DISTINCT ccId) > 1)",
  "edges with an endpoint missing from nodes.csv (expect 0)": "SELECT COUNT(*) FROM labeled_edges WHERE ccId IS NULL OR ccId_end IS NULL",
  "edges crossing components (expect 0)": "SELECT COUNT(*) FROM labeled_edges WHERE ccId <> ccId_end",
  "components with no nodes (expect 0)": "SELECT COUNT(*) FROM components WHERE ccId NOT IN (SELECT ccId FROM nodes)",
}
for name, q in checks.items():
    print(f"{name}: {con.sql(q).fetchone()[0]:,}")
con.sql("SELECT ccLabel, COUNT(*) AS n FROM components GROUP BY ccLabel").show()

components (expect 121,810): 121,810
rows in nodes.csv (expect 444,521): 444,521
nodes after feature join (should match): 444,521
rows in edges.csv (expect 367,137): 367,137
edges after feature join (should match): 367,137
duplicate clId in node_features (expect 0): 0
clId in more than one component (expect 0): 0
edges with an endpoint missing from nodes.csv (expect 0): 0
edges crossing components (expect 0): 0
components with no nodes (expect 0): 0
┌────────────┬────────┐
│  ccLabel   │   n    │
│  varchar   │ int64  │
├────────────┼────────┤
│ licit      │ 119047 │
│ suspicious │   2763 │
└────────────┴────────┘



**Result**: All checks passed. The counts are 121,810 components (119,047 licit and 2,763 suspicious), 444,521 nodes, and 367,137 edges, the same before and after the feature joins. There are no duplicates, no missing endpoints, no edges crossing components, and no components without nodes.

### **8. Save the results**

**Goal**: Save the four tables as compressed Parquet files so everyone can use them without re-running the big scans.

In [8]:
for t in ["labeled_nodes", "labeled_edges", "node_features", "edge_features"]:
    con.sql(f"COPY {t} TO '{OUT}/{t}.parquet' (FORMAT parquet, COMPRESSION zstd)")

**Result**: Four files in the output folder 
- `labeled_nodes.parquet`
- `labeled_edges.parquet`
- `node_features.parquet`
- `edge_features.parquet`